# Building a sample dataset for testing derivative products across different Landsat sensors

This notebook is designed to create a sample set for testing derivative products across the continent, where the testing needs to compare data derived from multiple landsat sensors. It is designed to do the following:

- Import a geojson containing the footprints of the 'golden tiles' that have been previously used for validating GeoMAD products. These tiles are distributed across Australia and across a range of environment types.
- Load a datacube dataset for the selected sensors (in this case, Landsat 7, and Landsat 8/9), using one or more of the golden tiles as the geometry for the datacube query
- Compare the resulting datasets and find the timesteps that overlap ( +/- a timeframe set by the user, e.g. 48 hours)
- If the resulting filtered dataset is very large, create a subset based on randomly selecting smaller regions or pixels
- Export the resulting geojson so it can be used as an input to test workflows

In [1]:
import os
import datacube
import pandas as pd
import numpy as np
import xarray as xr
import geopandas as gpd
import pprint
from datetime import timedelta
import matplotlib.pyplot as plt
from datacube.utils.geometry import CRS, Geometry, GeoBox
from datacube.utils import masking
from pathlib import Path
from shapely.geometry import box

import odc.geo.xr
from odc.geo.xr import assign_crs
from odc.io.cgroups import get_cpu_quota

import sys

sys.path.insert(1, ".../Tools")
from dea_tools.datahandling import load_ard
from dea_tools.classification import collect_training_data
from dea_tools.dask import create_local_dask_cluster
from dea_tools.plotting import rgb, display_map
from dea_tools.spatial import xr_vectorize, xr_rasterize
from dea_tools.bandindices import calculate_indices

import warnings
warnings.filterwarnings("ignore")

In [2]:
def select_tile(grid_gdf, region_code, query):
    region_code = [region_code]
    gdf = grid_gdf[grid_gdf["region_code"].isin(region_code)]
    polygon = gdf.geometry.iloc[0]

    geom = Geometry(geom=polygon, crs=gdf.crs)
    query.update({"geopolygon": geom})

    return query


In [3]:
# Example: Search for a file named 'target_file.txt' in the current directory and all subdirectories

target_filename = "testing_tile_suite.geojson"
for root, dirs, files in os.walk("."):
    if target_filename in files:
        file_path = Path(root) / target_filename
        print(f"Found: {file_path.resolve()}")


Found: /home/jovyan/dev/development_notebooks_JAG/01_projects/tassel_cap_exploration/testing_tile_suite.geojson


In [4]:
test_tiles_gdf = gpd.read_file(file_path.resolve())

# Extract the list of region_codes
region_codes = test_tiles_gdf["region_code"].tolist()

# remove shortlist once testing is done
region_codes = "x57y30"
# region_codes = region_codes[1:2]
pprint.pprint(region_codes)


'x57y30'


In [5]:
test_dir = "sample_points_abares.gpkg"
test_sample = gpd.read_file(test_dir)

In [6]:
test_sample.head()

,spatial_ref,class,geometry
0,3577,400,POINT (1128705.000 -3946875.000)
1,3577,400,POINT (1080015.000 -3954015.000)
2,3577,400,POINT (1068915.000 -4030935.000)
3,3577,400,POINT (1065405.000 -3957165.000)
4,3577,400,POINT (1106445.000 -3986805.000)


In [7]:
# modify query to use bounding box of points as geometry bounds
xmin, ymin, xmax, ymax = test_sample.total_bounds
polygon = box(xmin, ymin, xmax, ymax)

geom = Geometry(geom=polygon, crs=test_sample.crs)


In [8]:
# set up baseline dc query. THis will be modified by functions as needed later on.

query = {
    "time": ("2020-02", "2020-03"),
    "resolution": (-30, 30),
    "output_crs": "EPSG:3577",
    "group_by": "solar_day",
    "geopolygon": geom
}

In [9]:
# # for selecting the continent-wide sample data we will stick with landsat 8 for now.
# ds = load_ard(
#     dc=dc,
#     products=["ga_ls8c_ard_3", "ga_ls9c_ard_3"],
#     measurements=["nbart_blue", "nbart_green", "nbart_red", "nbart_nir", "nbart_swir_1", "nbart_swir_2", "oa_fmask", "oa_nbart_contiguity"],
#     cloud_mask="fmask",
#     mask_pixel_quality=True,
#     mask_contiguity=True,
#     dask_chunks={},
#     **query
# )

# attrs = ds.attrs

In [10]:
def feature_layers(query):
    dc = datacube.Datacube()

    ds = load_ard(
    dc=dc,
    products=["ga_ls8c_ard_3", "ga_ls9c_ard_3"],
    measurements=["nbart_blue", "nbart_green", "nbart_red", "nbart_nir", "nbart_swir_1", "nbart_swir_2", "oa_fmask", "oa_nbart_contiguity"],
    cloud_mask="fmask",
    mask_pixel_quality=True,
    mask_contiguity=True,
    **query)

    da = calculate_indices(ds,
                          index=["TCW", "TCB", "TCG"],
                          drop=False,
                          collection = "ga_ls_3")

    return da

In [11]:
if get_cpu_quota() is not None:
    ncpus = round(get_cpu_quota())
else:
    ncpus = os.cpu_count()
print(f"ncpus = {ncpus}")

ncpus = 15


In [12]:
column_names, model_input = collect_training_data(
    gdf = test_sample,
    dc_query=query,
    ncpus = ncpus,
    return_coords=True,
    field='class',
    zonal_stats='mean', #not actually going to use this but it complains if set to False.
    feature_func = feature_layers
)

Taking zonal statistic: mean


  0%|          | 0/500 [00:00<?, ?it/s]

Finding datasets
    ga_ls8c_ard_3
Finding datasets
Finding datasetsFinding datasets    ga_ls8c_ard_3Finding datasets

Finding datasets
    ga_ls8c_ard_3

    ga_ls8c_ard_3    ga_ls8c_ard_3
Finding datasets
    ga_ls8c_ard_3


    ga_ls8c_ard_3Finding datasetsFinding datasets


    ga_ls8c_ard_3    ga_ls8c_ard_3Finding datasets


    ga_ls8c_ard_3
Finding datasetsFinding datasets
Finding datasets
    ga_ls8c_ard_3
    ga_ls8c_ard_3
    ga_ls8c_ard_3

Finding datasets
    ga_ls8c_ard_3
Finding datasets
    ga_ls8c_ard_3
    ga_ls9c_ard_3
    ga_ls9c_ard_3
    ga_ls9c_ard_3
Applying fmask pixel quality/cloud mask
Applying contiguity mask (oa_nbart_contiguity)
Returning 4 time steps as a dask array
    ga_ls9c_ard_3
    ga_ls9c_ard_3
    ga_ls9c_ard_3
    ga_ls9c_ard_3
    ga_ls9c_ard_3    ga_ls9c_ard_3

    ga_ls9c_ard_3
    ga_ls9c_ard_3    ga_ls9c_ard_3

Applying fmask pixel quality/cloud mask
Applying contiguity mask (oa_nbart_contiguity)
Applying fmask pixel quality/cloud mask
Applyi

Error opening source dataset: s3://dea-public-data/baseline/ga_ls8c_ard_3/093/085/2020/02/07/ga_ls8c_nbart_3-1-0_093085_2020-02-07_final_band06.tif


Percentage of possible fails after run 1 = 0.0 %
Removed 0 rows wth NaNs &/or Infs
Output shape:  (499, 14)


In [13]:
model_input

array([[ 4.00000000e+02,  9.88250000e+02,  1.35450000e+03, ...,
         4.31074724e-02,  1.12870500e+06, -3.94687500e+06],
       [ 4.00000000e+02,  7.89333313e+02,  1.15466663e+03, ...,
         1.26216235e-02,  1.14883500e+06, -4.01863500e+06],
       [ 4.00000000e+02,  1.04925000e+03,  1.36950000e+03, ...,
         1.40263913e-02,  1.14877500e+06, -3.98062500e+06],
       ...,
       [ 2.00000000e+02,  7.52000000e+02,  9.51666687e+02, ...,
         6.46114722e-02,  1.10059500e+06, -3.93883500e+06],
       [ 2.00000000e+02,  1.11625000e+03,  1.44625000e+03, ...,
         3.08322273e-02,  1.13647500e+06, -3.98362500e+06],
       [ 1.00000000e+02,  7.14000000e+02,  8.88799988e+02, ...,
         4.84362356e-02,  1.08706500e+06, -3.93610500e+06]])

In [14]:
output_file = "stratified_sampling_abares_tassel_caps.txt"

In [15]:
np.savetxt(output_file, model_input, header = " ".join(column_names), fmt="%4f")